# Withings API

In [10]:
import pandas as pd
from pprint import pprint
from datetime import timedelta, date, timezone, datetime
from withings_api import WithingsAuth, WithingsApi
from withings_api.common import AuthScope, MeasureType, GetSleepSummaryField, MeasureGetMeasGroupCategory, GetActivityField
from urllib.parse import urlparse, parse_qs

In [ ]:
# Replace these with your own credentials
CLIENT_ID = 
CLIENT_SECRET = 
CALLBACK_URI = "http://localhost:5000/get_token"

In [12]:
auth = WithingsAuth(
    client_id=CLIENT_ID,
    consumer_secret=CLIENT_SECRET,
    callback_uri=CALLBACK_URI,
    scope=(
        AuthScope.USER_ACTIVITY,
        AuthScope.USER_METRICS,
        AuthScope.USER_INFO
    )
)

authorize_url = auth.get_authorize_url()

print("Go to this URL and authorize:")
print(authorize_url)

Go to this URL and authorize:
https://account.withings.com/oauth2_user/authorize2?response_type=code&client_id=ecf78d7a1c05d75c58b78bd7f234644a1bf5f6b9e8e47b2bb74021346620442e&redirect_uri=http%3A%2F%2Flocalhost%3A5000%2Fget_token&scope=user.activity%2Cuser.metrics%2Cuser.info&state=nKKbrtgTCDkm0Lgp41ehdZQcGFnf0w


In [15]:
code = "http://localhost:5000/get_token?code=342f58490193cb3a2c1590287c8157d929a166f8"

In [16]:
redirected_url = code.strip()
code = parse_qs(urlparse(redirected_url).query)["code"][0]

creds = auth.get_credentials(code)
api = WithingsApi(creds)

In [ ]:
[x for x in dir(api)]

['PATH_MEASURE',
 'PATH_NOTIFY',
 'PATH_V2_HEART',
 'PATH_V2_MEASURE',
 'PATH_V2_SLEEP',
 'PATH_V2_USER',
 'URL',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_blank_refresh_cb',
 '_client',
 '_credentials',
 '_refresh_cb',
 '_request',
 '_update_token',
 'get_credentials',
 'heart_get',
 'heart_list',
 'measure_get_activity',
 'measure_get_meas',
 'notify_get',
 'notify_list',
 'notify_revoke',
 'notify_subscribe',
 'notify_update',
 'refresh_token',
 'request',
 'sleep_get',
 'sleep_get_summary',
 'user_get_device']

In [17]:
import arrow
import pandas as pd

def heart_list_to_df(api, start_dt, end_dt, tz="Australia/ACT"):
    """
    Fetch Heart v2 records (ECG / Afib / heart events) without triggering pydantic validation,
    then return a DataFrame.
    """
    params = {
        "action": "list",
        "startdate": arrow.get(start_dt).int_timestamp,
        "enddate": arrow.get(end_dt).int_timestamp,
    }

    raw = api.request(path=api.PATH_V2_HEART, params=params)
    body = raw.get("body", {})
    series = body.get("series", [])

    rows = []
    for item in series:
        # date is usually unix timestamp (seconds)
        ts = item.get("date")
        dt = arrow.get(ts).to(tz).datetime if ts is not None else None

        rows.append({
            "datetime": dt,
            "date": dt.date() if dt else None,

            # common heart-list fields (some may be missing)
            "id": item.get("id"),
            "model_raw": item.get("model"),          # can be 94 etc.
            "deviceid": item.get("deviceid"),

            "heart_rate": item.get("heart_rate"),    # may be missing!
            "afib": item.get("afib"),                # may exist
            "ecg": item.get("ecg"),                  # may exist (often 1/0)
            "comment": item.get("comment"),
        })

    # df = pd.DataFrame(rows).sort_values("datetime").reset_index(drop=True)
    return df, raw


In [18]:
end_dt = arrow.utcnow()
start_dt = end_dt.shift(days=-90)
params = {
        "action": "list",
        "startdate": arrow.get(start_dt).int_timestamp,
        "enddate": arrow.get(end_dt).int_timestamp,
        "more": True
    }
api.request(path=api.PATH_V2_HEART, params=params)

{'series': [], 'more': False, 'offset': 0}

In [19]:
import arrow
import pandas as pd

def heart_list(api, start_dt, end_dt, offset=0):
    params = {
        "action": "list",
        "startdate": arrow.get(start_dt).int_timestamp,
        "enddate": arrow.get(end_dt).int_timestamp,
        "offset": offset,
    }
    return api.request(path=api.PATH_V2_HEART, params=params)

def heart_get_ecg_detail(api, signalid):
    # Heart v2 get uses action=get with signalid for ECG entries
    params = {"action": "get", "signalid": int(signalid)}
    return api.request(path=api.PATH_V2_HEART, params=params)

# 1) Get list
end_dt = arrow.utcnow()
start_dt = end_dt.shift(days=-90)
raw = heart_list(api, start_dt, end_dt)

series = raw.get("series", [])
print("records:", len(series), "more:", raw.get("more"), "offset:", raw.get("offset"))

# 2) Fetch detail for each ECG signal
details = []
for item in series:
    ecg = item.get("ecg") or {}
    signalid = ecg.get("signalid")
    if signalid is None:
        continue

    det = heart_get_ecg_detail(api, signalid)
    details.append({
        "timestamp": item.get("timestamp"),
        "heart_rate": item.get("heart_rate"),
        "model": item.get("model"),
        "signalid": signalid,
        "afib": ecg.get("afib"),
        "detail_raw": det,  # keep raw; we can flatten later
    })

df_details = pd.DataFrame(details)
df_details.head()

records: 0 more: False offset: 0


""


In [20]:
import arrow
import pandas as pd

end_dt = arrow.utcnow()
start_dt = end_dt.shift(days=-90)

resp = api.measure_get_activity(
    startdateymd=start_dt,
    enddateymd=end_dt,
    lastupdate=None,
    data_fields=[
        GetActivityField.STEPS,
        GetActivityField.DISTANCE,
        GetActivityField.CALORIES,
        GetActivityField.TOTAL_CALORIES,   # or TOTALCALORIES depending on enum
        GetActivityField.HR_AVERAGE,
        GetActivityField.HR_MIN,
        GetActivityField.HR_MAX,
        GetActivityField.HR_ZONE_0,
        GetActivityField.HR_ZONE_1,
        GetActivityField.HR_ZONE_2,
        GetActivityField.HR_ZONE_3,
    ],
)

# resp.activities is a list of MeasureGetActivityActivity objects
rows = []
for a in resp.activities:
    rows.append({
        "date": a.date.to("Australia/ACT").date(),
        "hr_avg": getattr(a, "hr_average", None),
        "hr_min": getattr(a, "hr_min", None),
        "hr_max": getattr(a, "hr_max", None),
        "hr_zone_0_sec": getattr(a, "hr_zone_0", None),
        "hr_zone_1_sec": getattr(a, "hr_zone_1", None),
        "hr_zone_2_sec": getattr(a, "hr_zone_2", None),
        "hr_zone_3_sec": getattr(a, "hr_zone_3", None),
    })

df_hr_daily = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
df_hr_daily.tail(10)


,date,hr_avg,hr_min,hr_max,hr_zone_0_sec,hr_zone_1_sec,hr_zone_2_sec,hr_zone_3_sec
42,2026-09-02,79.0,44.0,111.0,53200.0,7437.0,0.0,0.0
43,2026-09-03,75.0,47.0,107.0,40812.0,4231.0,0.0,0.0
44,2026-09-04,86.0,44.0,140.0,47817.0,14495.0,124.0,0.0
45,2026-09-05,77.0,36.0,181.0,53402.0,3973.0,290.0,67.0
46,2026-09-06,79.0,37.0,150.0,62199.0,9413.0,428.0,0.0
47,2026-09-07,57.0,57.0,57.0,37.0,0.0,0.0,0.0
48,2026-09-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,2026-09-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50,2026-09-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,2026-09-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
import withings_api
[x for x in dir(withings_api) if "Activity" in x or "MeasureGetActivity" in x]

['GetActivityField', 'MeasureGetActivityResponse']

## Extract Sleep Data

In [ ]:
end_dt = datetime.now(timezone.utc).date()
start_dt = (datetime.now(timezone.utc) - timedelta(days=90)).date()

sleep_summary = api.sleep_get_summary(
    startdateymd=start_dt,
    enddateymd=end_dt,
    lastupdate=start_dt,
    data_fields=[
        GetSleepSummaryField.SLEEP_SCORE,
        GetSleepSummaryField.WAKEUP_COUNT,
        GetSleepSummaryField.DEEP_SLEEP_DURATION,
        GetSleepSummaryField.LIGHT_SLEEP_DURATION,
        GetSleepSummaryField.REM_SLEEP_DURATION,
        GetSleepSummaryField.WAKEUP_DURATION,
        GetSleepSummaryField.HR_AVERAGE,
        GetSleepSummaryField.HR_MIN,
        GetSleepSummaryField.HR_MAX,
        GetSleepSummaryField.RR_AVERAGE,
        GetSleepSummaryField.RR_MAX,
        GetSleepSummaryField.RR_MIN,
        GetSleepSummaryField.SNORING,
        GetSleepSummaryField.SNORING_EPISODE_COUNT
    ],
)

sleep_summary, len(sleep_summary.series)

(SleepGetSummaryResponse(more=False, offset=0, series=(GetSleepSummarySerie(timezone=tzfile('Australia/ACT'), model=<SleepModel.TRACKER: 16>, startdate=<Arrow [2026-01-29T12:22:00+11:00]>, enddate=<Arrow [2026-01-29T17:44:00+11:00]>, date=<Arrow [2026-01-29T11:00:00+11:00]>, modified=<Arrow [2026-01-31T08:02:42+11:00]>, data=GetSleepSummaryData(breathing_disturbances_intensity=None, deepsleepduration=7920, durationtosleep=None, durationtowakeup=None, hr_average=55, hr_max=79, hr_min=46, lightsleepduration=3060, remsleepduration=600, rr_average=14, rr_max=16, rr_min=12, sleep_score=43, snoring=None, snoringepisodecount=None, wakeupcount=1, wakeupduration=480), id=5945102582), GetSleepSummarySerie(timezone=tzfile('Australia/ACT'), model=<SleepModel.TRACKER: 16>, startdate=<Arrow [2026-01-28T21:24:00+11:00]>, enddate=<Arrow [2026-01-29T09:00:00+11:00]>, date=<Arrow [2026-01-29T11:00:00+11:00]>, modified=<Arrow [2026-02-07T07:18:17+11:00]>, data=GetSleepSummaryData(breathing_disturbances_i

In [ ]:
pprint(sleep_summary.series[0])

GetSleepSummarySerie(timezone=tzfile('Australia/ACT'), model=<SleepModel.TRACKER: 16>, startdate=<Arrow [2026-01-29T12:22:00+11:00]>, enddate=<Arrow [2026-01-29T17:44:00+11:00]>, date=<Arrow [2026-01-29T11:00:00+11:00]>, modified=<Arrow [2026-01-31T08:02:42+11:00]>, data=GetSleepSummaryData(breathing_disturbances_intensity=None, deepsleepduration=7920, durationtosleep=None, durationtowakeup=None, hr_average=55, hr_max=79, hr_min=46, lightsleepduration=3060, remsleepduration=600, rr_average=14, rr_max=16, rr_min=12, sleep_score=43, snoring=None, snoringepisodecount=None, wakeupcount=1, wakeupduration=480), id=5945102582)


In [ ]:
def sleep_summary_to_df(series, tz="Australia/ACT"):
    """
    Convert Withings sleep_get_summary response series into a flat pandas DataFrame.
    
    series: list[GetSleepSummarySerie]
    tz: timezone name for converting Arrow timestamps
    """

    rows = []
    for s in series:
        d = getattr(s, "data", None)

        # Safe-get Arrow fields then convert to python date/datetime
        date_val = getattr(s, "date", None)
        start_val = getattr(s, "startdate", None)
        end_val = getattr(s, "enddate", None)
        modified_val = getattr(s, "modified", None)

        rows.append({
            "date": date_val.to(tz).date() if date_val else None,
            "start_datetime": start_val.to(tz).datetime if start_val else None,
            "end_datetime": end_val.to(tz).datetime if end_val else None,
            "modified_datetime": modified_val.to(tz).datetime if modified_val else None,

            "sleep_model": getattr(s, "model", None).name if getattr(s, "model", None) else None,
            "sleep_id": getattr(s, "id", None),

            # requested data fields
            "sleep_score": getattr(d, "sleep_score", None) if d else None,
            "wakeup_count": getattr(d, "wakeupcount", None) if d else None,

            # optional extras (keep them; they’re often useful later)
            "wakeup_duration_sec": getattr(d, "wakeupduration", None) if d else None,
            "deepsleep_duration_sec": getattr(d, "deepsleepduration", None) if d else None,
            "lightsleep_duration_sec": getattr(d, "lightsleepduration", None) if d else None,
            "remsleep_duration_sec": getattr(d, "remsleepduration", None) if d else None,
            "hr_average": getattr(d, "hr_average", None) if d else None,
            "hr_min": getattr(d, "hr_min", None) if d else None,
            "hr_max": getattr(d, "hr_max", None) if d else None,
            "rr_average": getattr(d, "hr_max", None) if d else None,
            "rr_max": getattr(d, "rr_max", None) if d else None,
            "rr_min": getattr(d, "rr_min", None) if d else None,
            "snoring": getattr(d, "snoring", None) if d else None,
            "snoringepisodecount": getattr(d, "snoringepisodecount", None) if d else None,
        })

    df = pd.DataFrame(rows)

    # Sort and de-dup (you can change this rule)
    df = df.sort_values(["date", "start_datetime"]).reset_index(drop=True)

    # If multiple sleep entries exist for the same date, you may want the "main sleep"
    # For now: keep all rows. If you want one row per date, tell me and I’ll collapse it.
    return df


In [ ]:
df_sleep = sleep_summary_to_df(sleep_summary.series)
df_sleep.tail(10)

,date,start_datetime,end_datetime,modified_datetime,sleep_model,sleep_id,sleep_score,wakeup_count,wakeup_duration_sec,deepsleep_duration_sec,lightsleep_duration_sec,remsleep_duration_sec,hr_average,hr_min,hr_max,rr_average,rr_max,rr_min,snoring,snoringepisodecount
72,2026-02-28,2026-02-28 00:28:00+11:00,2026-02-28 09:26:00+11:00,2026-03-02 20:47:28+11:00,TRACKER,6013657642,69,3,2340,8640,15360,5940,58,52,76,76,18.0,13.0,None,None
73,2026-03-01,2026-03-01 00:54:00+11:00,2026-03-01 16:03:00+11:00,2026-03-02 14:10:41+11:00,TRACKER,6014988596,72,8,8040,11100,20160,1860,64,37,114,114,21.0,8.0,None,None
74,2026-03-02,2026-03-01 21:36:00+11:00,2026-03-02 08:50:00+11:00,2026-03-03 18:17:03+11:00,TRACKER,6017852957,82,3,3720,11220,20340,5160,55,48,82,82,22.0,11.0,None,None
75,2026-03-02,2026-03-02 15:23:00+11:00,2026-03-02 17:10:00+11:00,2026-03-07 17:41:06+11:00,TRACKER,6018406518,20,1,780,3420,1680,540,72,58,121,121,17.0,12.0,None,None
76,2026-03-03,2026-03-02 22:52:00+11:00,2026-03-03 06:15:00+11:00,2026-03-04 16:28:47+11:00,TRACKER,6020916851,78,4,4380,8160,10740,3300,53,48,71,71,19.0,11.0,None,None
77,2026-03-04,2026-03-03 23:08:00+11:00,2026-03-04 09:04:00+11:00,2026-03-07 17:41:07+11:00,TRACKER,6022974071,80,2,1080,11400,17280,6000,58,50,104,104,20.0,10.0,None,None
78,2026-03-05,2026-03-04 21:46:00+11:00,2026-03-05 07:28:00+11:00,2026-03-07 17:41:08+11:00,TRACKER,6025355721,78,3,1980,10080,20400,2460,59,36,101,101,18.0,10.0,None,None
79,2026-03-06,2026-03-05 22:12:00+11:00,2026-03-06 09:33:00+11:00,2026-03-07 17:41:09+11:00,TRACKER,6027488676,66,7,6000,10500,17040,7320,61,36,168,168,22.0,9.0,None,None
80,2026-03-07,2026-03-06 23:48:00+11:00,2026-03-07 09:30:00+11:00,2026-03-07 17:34:54+11:00,TRACKER,6029908909,75,4,3600,8880,18780,3660,57,41,115,115,25.0,14.0,None,None
81,2026-03-07,2026-03-07 13:55:00+11:00,2026-03-07 15:54:00+11:00,2026-03-07 17:35:52+11:00,TRACKER,6029908949,26,2,1980,2640,2520,0,64,56,81,81,20.0,16.0,None,None


## Extract activities 
Fields include total and active calories

In [22]:
end = datetime.now(timezone.utc).date()
start = (datetime.now(timezone.utc) - timedelta(days=360)).date()  # last 90 days

resp = api.measure_get_activity(
    startdateymd=start,
    enddateymd=end,
)

len(resp.activities)

249

In [23]:
resp.activities

(MeasureGetActivityActivity(date=<Arrow [2025-11-05T00:00:00+00:00]>, timezone=tzfile('Australia/ACT'), deviceid=None, brand=1, is_tracker=False, steps=0, distance=0.0, elevation=0.0, soft=0, moderate=0, intense=0, active=0, calories=0.0, totalcalories=1894.23, hr_average=None, hr_min=None, hr_max=None, hr_zone_0=None, hr_zone_1=None, hr_zone_2=None, hr_zone_3=None),
 MeasureGetActivityActivity(date=<Arrow [2025-11-06T00:00:00+00:00]>, timezone=tzfile('Australia/ACT'), deviceid=None, brand=1, is_tracker=False, steps=0, distance=0.0, elevation=0.0, soft=0, moderate=0, intense=0, active=0, calories=0.0, totalcalories=1894.23, hr_average=None, hr_min=None, hr_max=None, hr_zone_0=None, hr_zone_1=None, hr_zone_2=None, hr_zone_3=None),
 MeasureGetActivityActivity(date=<Arrow [2025-11-07T00:00:00+00:00]>, timezone=tzfile('Australia/ACT'), deviceid=None, brand=1, is_tracker=False, steps=0, distance=0.0, elevation=0.0, soft=0, moderate=0, intense=0, active=0, calories=0.0, totalcalories=1894.23

In [24]:
rows = []

for a in resp.activities:
    soft = getattr(a, "soft", 0) or 0
    moderate = getattr(a, "moderate", 0) or 0
    intense = getattr(a, "intense", 0) or 0

    active_minutes = sum([soft, moderate, intense])

    # Convert Arrow date to Python date if present
    date_val = getattr(a, "date", None)
    if date_val is not None:
        date_val = date_val.date()

    rows.append({
        "date": date_val,
        "steps": getattr(a, "steps", None),

        "distance_m": getattr(a, "distance", None),
        "distance_km": (getattr(a, "distance", 0) or 0) / 1000.0,

        "active_minutes": active_minutes,
        "soft_minutes": soft,
        "moderate_minutes": moderate,
        "intense_minutes": intense,

        # Correct semantic names based on Withings API
        "active_burned_calories": getattr(a, "calories", None),
        "tdee_calories": getattr(a, "totalcalories", None),
    })

df_act = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

df_act.tail(10)

,date,steps,distance_m,distance_km,active_minutes,soft_minutes,moderate_minutes,intense_minutes,active_burned_calories,tdee_calories
239,2026-07-02,0,0.0,0.0,0,0,0,0,0.0,0.0
240,2026-07-03,0,0.0,0.0,0,0,0,0,0.0,0.0
241,2026-07-04,0,0.0,0.0,0,0,0,0,0.0,0.0
242,2026-07-05,0,0.0,0.0,0,0,0,0,0.0,0.0
243,2026-07-06,0,0.0,0.0,0,0,0,0,0.0,0.0
244,2026-07-07,0,0.0,0.0,0,0,0,0,0.0,0.0
245,2026-07-08,0,0.0,0.0,0,0,0,0,0.0,0.0
246,2026-07-09,0,0.0,0.0,0,0,0,0,0.0,0.0
247,2026-07-10,0,0.0,0.0,0,0,0,0,0.0,0.0
248,2026-07-11,0,0.0,0.0,0,0,0,0,0.0,0.0


In [ ]:
calories_cleaned = df_act[df_act['active_burned_calories'] ]
calories_cleaned.tail(10)

,date,steps,distance_m,distance_km,active_minutes,soft_minutes,moderate_minutes,intense_minutes,active_burned_calories,tdee_calories
79,2026-02-24,8081,8446.47,8.44647,10572,5881,1020,3671,509.50,2439.73
80,2026-02-25,7606,6170.72,6.17072,14457,8169,386,5902,791.26,2721.49
81,2026-02-26,5775,4676.87,4.67687,10341,7560,423,2358,320.35,2250.58
82,2026-02-27,3315,2610.30,2.61030,11295,10815,480,0,102.09,2032.32
83,2026-02-28,6982,5684.66,5.68466,18700,6179,6468,6053,795.35,2725.58
84,2026-03-01,3152,2480.33,2.48033,10528,6239,420,3869,689.30,2619.53
85,2026-03-02,6760,5477.36,5.47736,11530,6061,360,5109,629.27,2546.00
86,2026-03-03,13338,13589.48,13.58948,12479,6240,2459,3780,694.98,2611.71
87,2026-03-04,6246,6205.09,6.20509,12998,6542,300,6156,1155.24,3071.97
89,2026-03-06,2996,2380.91,2.38091,10999,6720,366,3913,695.74,2612.47


In [ ]:
calories_cleaned.to_csv('withings-activity-calories.csv', index=False)

In [ ]:
# Import MyFitnessPal data for comparison
mfp_data = pd.read_csv(r"C:\Users\jarro\Downloads\2026-02-01_to_2026-02-10-mfp.csv")
mfp_data = mfp_data.dropna() # drop the empty snack rows
mfp_data.head()

,Date,Meal,Food,Calories,Carbs,Fat,Protein,Sodium,Sugar
1,02/02/2026,Breakfast\n \n ...,"Egg, 5 large",358.0,2\n 2,24\n 62,31\n 36,355,1.0
2,02/02/2026,Breakfast\n \n ...,"Parmesan cheese, 1 tbsp",21.0,1\n 12,1\n 60,1\n 28,88,0.0
3,02/02/2026,Breakfast\n \n ...,"Home - Oat Milk Flat White, 1 Mug",187.0,0\n 0,6\n 81,3\n 19,0,15.0
4,02/02/2026,Lunch,"Strength Meals Co - Chicken & Potato Salad, 1 ...",424.0,34\n 33,15\n 32,36\n 35,"1,300",17.0
5,02/02/2026,Dinner,"Kirkland - Marinated Beef Bulgolgi, 200 g",274.0,14\n 21,10\n 32,32\n 47,800,4.0


In [ ]:
mfp_calories = (
    mfp_data.groupby("Date", as_index=False)["Calories"].sum()
)

mfp_calories["Date"] = pd.to_datetime(mfp_calories["Date"], format="%d/%m/%Y").dt.date
mfp_calories = mfp_calories.rename(
    columns={"Calories": "consumed_calories",
             "Date": "date"}
)
mfp_calories

,date,consumed_calories
0,2026-02-02,1990.0
1,2026-02-03,1946.0
2,2026-02-04,2264.0
3,2026-02-05,2350.0
4,2026-02-06,1736.0
5,2026-02-07,1876.0
6,2026-02-08,2130.0
7,2026-02-09,2023.0
8,2026-02-10,1846.0


In [ ]:
merged_df = df_act.merge(mfp_calories, how='right')
merged_df

,date,steps,distance_m,distance_km,active_minutes,soft_minutes,moderate_minutes,intense_minutes,active_burned_calories,tdee_calories,consumed_calories
0,2026-02-02,2804,2330.46,2.33046,10539,6602,480,3457,613.34,2516.07,1990.0
1,2026-02-03,6534,6336.17,6.33617,11416,6540,960,3916,776.50,2679.23,1946.0
2,2026-02-04,4266,3274.19,3.27419,12123,8865,354,2904,553.57,2456.30,2264.0
3,2026-02-05,5729,4424.36,4.42436,17860,16213,133,1514,401.30,2304.03,2350.0
4,2026-02-06,8169,6929.34,6.92934,14715,6645,2344,5726,692.37,2595.10,1736.0
5,2026-02-07,7990,7473.28,7.47328,11963,8881,900,2182,455.41,2358.14,1876.0
6,2026-02-08,5854,5739.41,5.73941,9297,6371,1500,1426,523.64,2422.87,2130.0
7,2026-02-09,9348,7533.81,7.53381,15377,8580,1928,4869,694.82,2594.05,2023.0
8,2026-02-10,7233,7588.92,7.58892,6783,3543,894,2346,449.95,2144.18,1846.0


In [ ]:
merged_df["net_balance"] = merged_df["consumed_calories"] - merged_df["tdee_calories"]
merged_df["deficit"] = merged_df["tdee_calories"] - merged_df["consumed_calories"]
merged_df

,date,steps,distance_m,distance_km,active_minutes,soft_minutes,moderate_minutes,intense_minutes,active_burned_calories,tdee_calories,consumed_calories,net_balance,deficit
0,2026-02-02,2804,2330.46,2.33046,10539,6602,480,3457,613.34,2516.07,1990.0,-526.07,526.07
1,2026-02-03,6534,6336.17,6.33617,11416,6540,960,3916,776.50,2679.23,1946.0,-733.23,733.23
2,2026-02-04,4266,3274.19,3.27419,12123,8865,354,2904,553.57,2456.30,2264.0,-192.30,192.30
3,2026-02-05,5729,4424.36,4.42436,17860,16213,133,1514,401.30,2304.03,2350.0,45.97,-45.97
4,2026-02-06,8169,6929.34,6.92934,14715,6645,2344,5726,692.37,2595.10,1736.0,-859.10,859.10
5,2026-02-07,7990,7473.28,7.47328,11963,8881,900,2182,455.41,2358.14,1876.0,-482.14,482.14
6,2026-02-08,5854,5739.41,5.73941,9297,6371,1500,1426,523.64,2422.87,2130.0,-292.87,292.87
7,2026-02-09,9348,7533.81,7.53381,15377,8580,1928,4869,694.82,2594.05,2023.0,-571.05,571.05
8,2026-02-10,7233,7588.92,7.58892,6783,3543,894,2346,449.95,2144.18,1846.0,-298.18,298.18


### Estimating weight loss per week and providing total summary

In [ ]:
def estimate_weight_loss_by_week(df, date_col="date", deficit_col="deficit", kcal_per_kg=7700):
    """
    Calculate weekly and total estimated weight loss from a dataframe.

    Returns:
        - weekly summary dataframe
        - totals dictionary
    """

    # Ensure date column is datetime
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # Group by calendar week (ISO year + week)
    weekly = (
        df.groupby(pd.Grouper(key=date_col, freq="W"))
          .agg(
              weekly_deficit_kcal=(deficit_col, "sum"),
              avg_daily_deficit_kcal=(deficit_col, "mean"),
              days=("deficit", "count")
          )
          .reset_index()
    )

    # Estimate kg loss per week
    weekly["estimated_kg_loss"] = weekly["weekly_deficit_kcal"] / kcal_per_kg
    weekly["estimated_lbs_loss"] = weekly["estimated_kg_loss"] * 2.20462

    weekly = weekly.rename(columns={"date": "week_ending"})

    # Totals across entire dataset
    total_deficit = df[deficit_col].sum()

    totals = {
        "total_deficit_kcal": total_deficit,
        "estimated_kg_loss": total_deficit / kcal_per_kg,
        "estimated_lbs_loss": (total_deficit / kcal_per_kg) * 2.20462,
        "weeks_of_data": len(weekly),
        "avg_weekly_deficit": weekly["weekly_deficit_kcal"].mean()
    }

    return weekly, totals


In [ ]:
weekly_summary, totals = estimate_weight_loss_by_week(merged_df)

pprint(totals)
weekly_summary

{'avg_weekly_deficit': 1954.4850000000001,
 'estimated_kg_loss': 0.5076584415584416,
 'estimated_lbs_loss': 1.1191939534285715,
 'total_deficit_kcal': 3908.9700000000003,
 'weeks_of_data': 2}


,week_ending,weekly_deficit_kcal,avg_daily_deficit_kcal,days,estimated_kg_loss,estimated_lbs_loss
0,2026-02-08,3039.74,434.248571,7,0.394771,0.870321
1,2026-02-15,869.23,434.615000,2,0.112887,0.248873


In [ ]:
STARTING_WEIGHT = 93.25
GOAL_WEIGHT = 85

In [ ]:
def project_goal_date(
    df,
    starting_weight,
    goal_weight,
    deficit_col="deficit",
    date_col="date",
    kcal_per_kg=7700
):
    """
    Projects when a goal weight will be reached based on historical deficit data.
    """

    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # Calculate historical averages
    total_deficit = df[deficit_col].sum()
    days = len(df)

    avg_daily_deficit = total_deficit / days

    # How many kg have been 'lost' so far (estimated)
    estimated_loss_so_far = total_deficit / kcal_per_kg

    # Current estimated weight
    current_estimated_weight = starting_weight - estimated_loss_so_far

    kg_remaining = current_estimated_weight - goal_weight

    if kg_remaining <= 0:
        return {
            "status": "Goal already achieved",
            "estimated_current_weight": round(current_estimated_weight, 2)
        }

    # Calories still required to lose
    kcal_remaining = kg_remaining * kcal_per_kg

    # Days required at current average pace
    days_to_goal = kcal_remaining / avg_daily_deficit

    projected_date = df[date_col].max() + timedelta(days=int(days_to_goal))

    return {
        "starting_weight": starting_weight,
        "goal_weight": goal_weight,
        "estimated_current_weight": round(current_estimated_weight, 2),
        "kg_remaining": round(kg_remaining, 2),
        "avg_daily_deficit": round(avg_daily_deficit, 2),
        "projected_days_to_goal": int(days_to_goal),
        "projected_goal_date": projected_date.date()
    }


In [ ]:
result = project_goal_date(
    merged_df,
    starting_weight=STARTING_WEIGHT,
    goal_weight=GOAL_WEIGHT
)

pprint(result)

{'avg_daily_deficit': 434.33,
 'estimated_current_weight': 92.74,
 'goal_weight': 85,
 'kg_remaining': 7.74,
 'projected_days_to_goal': 137,
 'projected_goal_date': datetime.date(2026, 6, 27),
 'starting_weight': 93.25}


# Exercise

In [35]:
import pandas as pd
import json
import time
from pathlib import Path
from tqdm.auto import tqdm

# Reuse your existing authenticated WithingsApi instance.
# Change this only if you named it something other than api.
client = api

START_DATE = "2026-04-01"
END_DATE = "2026-10-20"

OUTPUT_DIR = Path(r"C:\Users\jarro\Downloads")


In [26]:
ACTIVITY_FIELDS = [
    "steps",
    "distance",
    "elevation",
    "soft",
    "moderate",
    "intense",
    "active",
    "calories",
    "totalcalories",
    "hr_average",
    "hr_min",
    "hr_max",
    "hr_zone_0",
    "hr_zone_1",
    "hr_zone_2",
    "hr_zone_3",
]

WORKOUT_FIELDS = [
    "calories",
    "effduration",
    "intensity",
    "distance",
    "steps",
    "hr_average",
    "hr_min",
    "hr_max",
    "hr_zone_0",
    "hr_zone_1",
    "hr_zone_2",
    "hr_zone_3",
]

def fetch_withings(action, fields):
    records = []
    offset = None

    with tqdm(desc=f"Fetching {action}", unit="page") as pbar:
        while True:
            params = {
                "action": action,
                "startdateymd": START_DATE,
                "enddateymd": END_DATE,
                "data_fields": ",".join(fields),
            }

            if offset is not None:
                params["offset"] = offset

            result = client.request(
                path=client.PATH_V2_MEASURE,
                params=params,
            )

            key = "activities" if action == "getactivity" else "series"

            batch = result.get(key, [])
            records.extend(batch)

            pbar.update(1)
            pbar.set_postfix(rows=len(records))

            if not result.get("more"):
                break

            next_offset = result.get("offset")

            if next_offset is None or next_offset == offset:
                raise RuntimeError("Invalid pagination offset")

            offset = next_offset
            time.sleep(0.3)

    return records

In [27]:
activity_records = fetch_withings(
    action="getactivity",
    fields=ACTIVITY_FIELDS,
)

activity_df = pd.json_normalize(activity_records)

if not activity_df.empty:
    activity_df = activity_df.sort_values("date")
    activity_df.to_csv(
        OUTPUT_DIR / "withings_daily_activity.csv",
        index=False,
    )

print(f"Extracted {len(activity_df)} daily activity records")
display(activity_df.head(10))


Fetching getactivity: 0page [00:00, ?page/s]

Extracted 164 daily activity records


,steps,distance,elevation,soft,moderate,intense,active,calories,totalcalories,hr_average,...,hr_zone_3,deviceid,hash_deviceid,timezone,date,modified,brand,modelid,model,is_tracker
0,6365,5222.27,82.93,6600,6059,8,6067,1164.30,3098.53,71.0,...,222.0,None,None,Australia/Sydney,2026-04-01,1775170295,1,94,ScanWatch 2,True
1,2888,2297.55,2.76,7259,4592,0,4592,969.95,2904.18,68.0,...,0.0,None,None,Australia/Sydney,2026-04-02,1775170296,1,94,ScanWatch 2,True
2,2473,1957.39,7.45,6613,3045,10,3055,640.22,2574.45,71.0,...,0.0,None,None,Australia/Sydney,2026-04-03,1775256735,18,94,ScanWatch 2,True
3,5719,4478.67,25.48,9594,3070,22,3092,541.40,2475.63,76.0,...,83.0,None,None,Australia/Sydney,2026-04-04,1775371224,1,94,ScanWatch 2,True
4,12448,9830.73,120.61,6240,9462,3,9465,1772.94,3772.14,72.0,...,0.0,None,None,Australia/Sydney,2026-04-05,1775440402,1,94,ScanWatch 2,True
5,3458,2756.96,13.73,7128,4496,6,4502,893.43,2812.66,70.0,...,0.0,None,None,Australia/Sydney,2026-04-06,1775529316,1,94,ScanWatch 2,True
6,10421,10017.85,51.03,6301,1920,5437,7357,2196.19,4115.42,74.0,...,535.0,None,None,Australia/Sydney,2026-04-07,1775633349,1,94,ScanWatch 2,True
7,7118,5828.39,97.82,7333,5334,663,5997,1208.32,3127.55,79.0,...,0.0,None,None,Australia/Sydney,2026-04-08,1775703106,1,94,ScanWatch 2,True
8,3391,2686.23,11.17,8861,2185,2,2187,414.84,2334.07,78.0,...,642.0,None,None,Australia/Sydney,2026-04-09,1775848822,1,94,ScanWatch 2,True
9,3956,3119.81,13.70,8590,4704,0,4704,927.59,2846.82,70.0,...,0.0,None,None,Australia/Sydney,2026-04-10,1775890766,1,94,ScanWatch 2,True


## Workouts

In [28]:
workout_records = fetch_withings(
    action="getworkouts",
    fields=WORKOUT_FIELDS,
)

workouts_df = pd.json_normalize(workout_records)

if not workouts_df.empty:
    workouts_df = workouts_df.sort_values("startdate")

    # Convert Unix timestamps when present
    for col in ["startdate", "enddate"]:
        if col in workouts_df.columns:
            workouts_df[f"{col}_datetime"] = pd.to_datetime(
                workouts_df[col],
                unit="s",
                utc=True,
                errors="coerce",
            )

    workouts_df.to_csv(
        OUTPUT_DIR / "withings_workouts.csv",
        index=False,
    )

print(f"Extracted {len(workouts_df)} workouts")
display(workouts_df.head(10))


Fetching getworkouts: 0page [00:00, ?page/s]

Extracted 372 workouts


,id,category,timezone,model,attrib,startdate,enddate,date,deviceid,modified,...,data.hr_max,data.hr_zone_0,data.hr_zone_1,data.hr_zone_2,data.hr_zone_3,data.steps,data.distance,data.effduration,startdate_datetime,enddate_datetime
0,6086436040,16,Australia/Sydney,94,7,1775007478,1775011221,2026-04-01,None,1775022448,...,96.0,2957.0,80.0,0.0,0.0,668.0,531.900024,None,2026-04-01 01:37:58+00:00,2026-04-01 02:40:21+00:00
1,6090308949,1,Australia/Sydney,94,7,1775021329,1775023574,2026-04-01,fe174be50c207a2186ac1a52fa884ce5043bf596,1775256708,...,154.0,104.0,2038.0,65.0,0.0,4276.0,3544.610352,None,2026-04-01 05:28:49+00:00,2026-04-01 06:06:14+00:00
2,6090308963,16,Australia/Sydney,94,7,1775103420,1775107762,2026-04-02,None,1775170297,...,86.0,3035.0,0.0,0.0,0.0,685.0,539.619995,None,2026-04-02 04:17:00+00:00,2026-04-02 05:29:22+00:00
3,6092482407,16,Australia/Sydney,94,7,1775188553,1775191299,2026-04-03,None,1775256735,...,122.0,899.0,938.0,0.0,0.0,327.0,253.569992,None,2026-04-03 03:55:53+00:00,2026-04-03 04:41:39+00:00
4,6095046774,1,Australia/Sydney,94,0,1775269680,1775270400,2026-04-04,fe174be50c207a2186ac1a52fa884ce5043bf596,1775369084,...,NaN,NaN,NaN,NaN,NaN,929.0,696.020020,None,2026-04-04 02:28:00+00:00,2026-04-04 02:40:00+00:00
5,6095046798,1,Australia/Sydney,94,0,1775273580,1775274180,2026-04-04,fe174be50c207a2186ac1a52fa884ce5043bf596,1775369084,...,NaN,NaN,NaN,NaN,NaN,856.0,642.210022,None,2026-04-04 03:33:00+00:00,2026-04-04 03:43:00+00:00
6,6095046813,16,Australia/Sydney,94,7,1775276636,1775277883,2026-04-04,fe174be50c207a2186ac1a52fa884ce5043bf596,1775369085,...,NaN,NaN,NaN,NaN,NaN,246.0,196.250000,None,2026-04-04 04:23:56+00:00,2026-04-04 04:44:43+00:00
7,6095129011,1,Australia/Sydney,94,0,1775345700,1775349660,2026-04-05,fe174be50c207a2186ac1a52fa884ce5043bf596,1775371225,...,NaN,NaN,NaN,NaN,NaN,6688.0,5187.329590,None,2026-04-04 23:35:00+00:00,2026-04-05 00:41:00+00:00
8,6095046914,16,Australia/Sydney,94,7,1775362984,1775366555,2026-04-05,None,1775371226,...,102.0,1527.0,909.0,0.0,0.0,701.0,541.599976,None,2026-04-05 04:23:04+00:00,2026-04-05 05:22:35+00:00
9,6095129040,1,Australia/Sydney,94,7,1775369085,1775370717,2026-04-05,fe174be50c207a2186ac1a52fa884ce5043bf596,1775440384,...,135.0,33.0,1549.0,35.0,0.0,2942.0,2439.599854,None,2026-04-05 06:04:45+00:00,2026-04-05 06:31:57+00:00


In [29]:
if not activity_df.empty:
    summary = activity_df.copy()

    summary["date"] = pd.to_datetime(summary["date"])
    summary["week"] = summary["date"].dt.to_period("W-SUN").astype(str)

    metrics = [
        "steps",
        "distance",
        "calories",
        "totalcalories",
        "soft",
        "moderate",
        "intense",
        "active",
        "hr_average",
    ]

    available = [col for col in metrics if col in summary.columns]

    for col in available:
        summary[col] = pd.to_numeric(summary[col], errors="coerce")

    weekly_df = (
        summary.groupby("week")
        .agg(
            recorded_days=("date", "nunique"),
            **{
                f"avg_{col}": (col, "mean")
                for col in available
            }
        )
        .reset_index()
    )

    weekly_df.to_csv(
        OUTPUT_DIR / "withings_weekly_summary.csv",
        index=False,
    )

    display(weekly_df)

,week,recorded_days,avg_steps,avg_distance,avg_calories,avg_totalcalories,avg_soft,avg_moderate,avg_intense,avg_active,avg_hr_average
0,2026-03-30/2026-04-05,5,5978.600000,4757.322000,1017.762000,2964.986000,7261.200000,5245.600000,8.600000,5254.200000,71.600000
1,2026-04-06/2026-04-12,7,6027.857143,5077.228571,1142.041429,3065.057143,8245.285714,4153.285714,1019.000000,5172.285714,75.142857
2,2026-04-13/2026-04-19,7,5776.571429,4920.764286,945.100000,2867.687143,7956.571429,4464.285714,404.857143,4869.142857,74.571429
3,2026-04-20/2026-04-26,7,3690.428571,2913.532857,728.135714,2647.151429,7204.142857,3717.857143,89.571429,3807.428571,64.428571
4,2026-04-27/2026-05-03,7,5731.000000,4814.635714,811.482857,2712.998571,9022.714286,3052.714286,933.714286,3986.428571,74.285714
5,2026-05-04/2026-05-10,7,5962.142857,4726.788571,988.965714,2870.481429,7887.571429,4966.428571,160.428571,5126.857143,74.000000
6,2026-05-11/2026-05-17,7,8518.714286,6724.562857,1235.528571,3127.758571,12045.142857,4697.428571,1667.285714,6364.714286,78.285714
7,2026-05-18/2026-05-24,7,4118.571429,3272.285714,1073.984286,2974.142857,7389.428571,5143.714286,10.571429,5154.285714,72.285714
8,2026-05-25/2026-05-31,7,9720.571429,7571.035714,1204.158571,3094.102857,10451.285714,6618.285714,169.142857,6787.428571,77.000000
9,2026-06-01/2026-06-07,7,10994.142857,8636.995714,1199.341429,3088.142857,8850.571429,7192.571429,547.714286,7740.285714,77.428571


In [36]:
import zipfile

# Use your existing folder path
# output_dir = Path(OUTPUT_DIR)
# output_dir.mkdir(parents=True, exist_ok=True)

zip_path = OUTPUT_DIR / "withings_fitness_export.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for csv_file in OUTPUT_DIR.glob("*.csv"):
        z.write(csv_file, arcname=csv_file.name)

print(f"Export ready: {zip_path.resolve()}")


Export ready: C:\Users\jarro\Downloads\withings_fitness_export.zip


In [38]:
# Hotfix
from pathlib import Path
import zipfile

output_dir = Path(r"C:\Users\jarro\Downloads")
output_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "withings_daily_activity.csv": activity_df,
    "withings_workouts.csv": workouts_df,
    "withings_weekly_summary.csv": weekly_df,
}

zip_path = output_dir / "withings_fitness_export_NEW.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for filename, df in exports.items():
        if df is None or df.empty:
            print(f"Skipping {filename}: no data")
            continue

        csv_path = output_dir / filename
        df.to_csv(csv_path, index=False)
        z.write(csv_path, arcname=filename)

        print(f"{filename}: {len(df)} rows")

print(f"\nSaved: {zip_path}")


withings_daily_activity.csv: 164 rows
withings_workouts.csv: 372 rows
withings_weekly_summary.csv: 24 rows

Saved: C:\Users\jarro\Downloads\withings_fitness_export_NEW.zip
